# Music Annotation Project V8.8D
## Notebook 55: Notebook 09 Shadow Comparison

**Runtime:** CPU. Select **Runtime → Run all**.

This notebook compares the successfully reproduced Notebook 51R2 historical registered-replay path with the incumbent Notebook 09 path on the same page roster. The replay includes historical decision-code-2 exclusion solely because that is required to reproduce the registered 51R2 golden output. This does not authorise SAFE_PROTECT operationally.

### Entry conditions
- Notebook 53 passed.
- Notebook 54 passed the strict zero-tolerance equivalence audit.
- A completed Notebook 09 run exists for the same pages, or an explicit incumbent page directory is supplied.

### Fixed rules
- No retraining, recalibration, threshold movement or operational SAFE_PROTECT authorisation.
- Page-roster identity is mandatory.
- Outside-authority changes are forbidden.
- Disagreement is evidence for review, not permission to choose the more attractive output.
- Every failure is written before the notebook stops.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib pyarrow


In [ ]:
import matplotlib.pyplot as plt
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N53_ROOT=ROOT/'observed_print_classifier_v8_8_notebook53R3'
N54_ROOT=ROOT/'m5_remove_exact_equivalence_v8_8c'
N09_ROOT=ROOT/'production_pilot'/'runs'
OUTPUT_ROOT=ROOT/'m5_remove_notebook09_shadow_v8_8d'
NOTEBOOK53_RUN_OVERRIDE=None
NOTEBOOK54_RUN_OVERRIDE=None
NOTEBOOK09_RUN_OVERRIDE=None
INCUMBENT_PAGE_DIR_OVERRIDE=None
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook55_v8_8d_shadow'
OUT=OUTPUT_ROOT/RUN; T=OUT/'tables'; D=OUT/'differences'; V=OUT/'visual_review'; A=OUT/'audit'; U=OUT/'upload'
for p in [OUT,T,D,V,A,U]: p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)


## 1. Resolve and verify Notebook 53 and Notebook 54

In [ ]:
N53=Path(NOTEBOOK53_RUN_OVERRIDE) if NOTEBOOK53_RUN_OVERRIDE else latest_complete(
    N53_ROOT,
    [
        'stage_f_verdict.json',
        'experiment_identity.json',
        'tables/page_equivalence.csv',
        'tables/mask_inventory.csv',
        'tables/regional_masks.csv',
        'pages',
    ]
)

N54=Path(NOTEBOOK54_RUN_OVERRIDE) if NOTEBOOK54_RUN_OVERRIDE else latest_complete(
    N54_ROOT,
    ['notebook54_verdict.json']
)

require(N53 is not None, 'Complete Notebook 53R3 run not found.')
require(N54 is not None, 'Complete Notebook 54 run not found.')

v53=read_json(N53/'stage_f_verdict.json')
v54=read_json(N54/'notebook54_verdict.json')

require(
    v53.get('verdict') == 'PASS_STAGE_F_REGRESSION_COMPATIBILITY',
    'Notebook 53R3 did not pass regression compatibility.'
)

require(
    v54.get('verdict') ==
    'PASS_51R2_REGISTERED_REPLAY_EXACT_EQUIVALENCE_READY_FOR_NOTEBOOK09_SHADOW',
    'Notebook 54 did not pass registered-replay exact equivalence.'
)

id53=read_json(N53/'experiment_identity.json')

require(
    id53.get('version') ==
    '53R3-v8.8b-notebook51R2-regression-compatibility',
    'Unexpected Notebook 53R3 version.'
)
require(
    id53.get('mode') == 'REGISTERED_REPLAY',
    'Notebook 53R3 is not registered replay.'
)

source_version=id53['version']
source_mode=id53['mode']

print('N53R3:',N53)
print('N54:',N54)
print('Source version:',source_version)
print('Mode:',source_mode)

## 2. Resolve the incumbent Notebook 09 approved/final pages

In [ ]:
if INCUMBENT_PAGE_DIR_OVERRIDE is not None:
    incumbent_dir=Path(INCUMBENT_PAGE_DIR_OVERRIDE); N09=incumbent_dir.parent
else:
    N09=Path(NOTEBOOK09_RUN_OVERRIDE) if NOTEBOOK09_RUN_OVERRIDE else latest_complete(N09_ROOT,['pilot_manifest.json','final'])
    require(N09 is not None,'No completed Notebook 09 run found. Set NOTEBOOK09_RUN_OVERRIDE or INCUMBENT_PAGE_DIR_OVERRIDE.')
    incumbent_dir=N09/'final'
require(incumbent_dir.exists(),f'Incumbent page directory missing: {incumbent_dir}')
def page_files(folder): return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}}
incumbent=page_files(incumbent_dir)
v88={p.name: p/'corrected.png' for p in (N53/'pages').iterdir() if p.is_dir() and (p/'corrected.png').exists()}
roster53=set(v88); roster09=set(incumbent)
roster_audit={'notebook53_pages':len(roster53),'notebook09_pages':len(roster09),'only_notebook53':sorted(roster53-roster09),'only_notebook09':sorted(roster09-roster53),'exact':roster53==roster09}
atomic_json(A/'roster_audit.json',roster_audit)
require(roster53==roster09,f'Shadow roster mismatch: {roster_audit}')
pages=sorted(roster53); print('Shadow pages:',len(pages),'Notebook09:',N09)


## 3. Pixel-exact shadow comparison and disagreement localisation

In [ ]:
rows=[]
for page in pages:
    a=imread(incumbent[page])
    b=imread(v88[page])
    observed=imread(N53/'pages'/page/'observed.png')
    m3=imread(N53/'pages'/page/'m3.png')>0
    sr=imread(N53/'pages'/page/'safe_remove.png')>0
    sp=imread(N53/'pages'/page/'safe_protect_decision.png')>0
    m5=imread(N53/'pages'/page/'m5.png')>0

    require(a.shape==b.shape==observed.shape==m3.shape==sr.shape==m5.shape,f'{page}: shape mismatch')
    require(np.array_equal(m5, m3 & ~sr & ~sp),f'{page}: 53R3 registered-replay M5 contract failure')
    
    diff=a!=b
    absdiff=np.abs(a.astype(np.int16)-b.astype(np.int16))
    outside=int(np.sum(diff&~m3))
    inside=int(np.sum(diff&m3))
    on_sr=int(np.sum(diff&sr))
    on_m5=int(np.sum(diff&m5))
    
    d=D/page; 
    d.mkdir(parents=True,exist_ok=False)
    
    locator=np.zeros_like(a); locator[diff]=255; cv2.imwrite(str(d/'difference_locator.png'),locator)
    cv2.imwrite(str(d/'absolute_difference.png'),np.clip(absdiff,0,255).astype(np.uint8))
    overlay=cv2.cvtColor(observed,cv2.COLOR_GRAY2BGR) 
    overlay[diff]=(0,0,255)
    cv2.imwrite(str(d/'difference_overlay.png'),overlay)

    rows.append(
        {
            'page':page,
            'pixels':a.size,
            'different_pixels':int(diff.sum()),
            'different_fraction':float(diff.mean()),
            'max_absolute_difference':int(absdiff.max()),
            'mean_absolute_difference_on_diff':float(absdiff[diff].mean()) if diff.any() else 0.0,
            'outside_M3_disagreement':outside,
            'inside_M3_disagreement':inside,
            'SAFE_REMOVE_disagreement':on_sr,
            'SAFE_PROTECT_DECISION_disagreement':on_sp,
            'M5_REMOVE_disagreement':on_m5,
            'exact':bool(not diff.any()),
            'notebook09_sha256':array_hash(a),
            'v88_sha256':array_hash(b)
        }
    )

compare=pd.DataFrame(rows).sort_values(['different_pixels','page'],ascending=[False,True])
compare.to_csv(T/'shadow_page_comparison.csv',index=False)

require((compare.outside_M3_disagreement==0).all(),'Notebook09/V8.8 disagreement occurred outside M3; inspect audit before interpretation.')

print(
    compare[
        [
            'page',
            'different_pixels',
            'SAFE_REMOVE_disagreement',
            'SAFE_PROTECT_DECISION_disagreement',
            'M5_REMOVE_disagreement'
        ]
    ].head(20).to_string(index=False)
)


## 4. Review queues and contact sheets

In [ ]:
compare.assign(review_priority=np.where(compare.different_pixels>0,'REVIEW','NO_DIFFERENCE')).to_csv(T/'shadow_review_queue.csv',index=False)
def sheet(frame,name,n=12):
    q=frame.head(n)
    if q.empty:return
    fig,ax=plt.subplots(len(q),3,figsize=(15,max(5,5*len(q))),squeeze=False)
    for r,(_,row) in enumerate(q.iterrows()):
        page=row.page
        ims=[imread(incumbent[page]),imread(v88[page]),imread(D/page/'difference_overlay.png',cv2.IMREAD_COLOR)]
        titles=[f'{page} Notebook 09',f'{page} V8.8',f'{page} disagreement']
        for j,(im,title) in enumerate(zip(ims,titles)):
            if im.ndim==3: im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
            ax[r,j].imshow(im,cmap='gray' if im.ndim==2 else None,vmin=0,vmax=255); ax[r,j].set_title(title); ax[r,j].axis('off')
    plt.tight_layout(); plt.savefig(V/f'{name}.png',dpi=120,bbox_inches='tight'); plt.close(fig)
sheet(compare[compare.different_pixels>0],'largest_shadow_disagreements')
sheet(compare[compare.different_pixels==0],'exact_shadow_matches')


## 5. Fail-closed shadow verdict

In [ ]:
gates={'notebook53_pass':True,'notebook54_pass':True,'roster_exact':roster53==roster09,'all_pages_compared':len(compare)==len(pages),'outside_m3_disagreement_zero':bool((compare.outside_M3_disagreement==0).all()),'all_outputs_finite':True}
first_failed=next((k for k,v in gates.items() if v is not True),None)
verdict='PASS_REGISTERED_REPLAY_NOTEBOOK09_SHADOW_EVIDENCE_READY_FOR_SUPERVISED_PILOT' if first_failed is None else 'FAIL_V8_8D_REGISTERED_REPLAY_NOTEBOOK09_SHADOW'
record={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'verdict':verdict,
    'first_failed_gate':first_failed,
    'gates':gates,
    'notebook53_version': source_version,
    'notebook53_mode': source_mode,
    'historical_registered_replay': True,
    'historical_code2_exclusion_replayed': True,
    'safe_protect_operationally_authorised': False,
    'automatic_output_authority':False,
    'meaning':'PASS permits Notebook 56 supervised pilot. Notebook 09 remains decision authority.'
}

atomic_json(OUT/'notebook55_verdict.json',record); print(json.dumps(record,indent=2))

if first_failed: raise RuntimeError(f'Notebook 55 failed at {first_failed}')


## 6. Curated upload package

In [ ]:
summary=['NOTEBOOK 55 / V8.8D NOTEBOOK 09 SHADOW COMPARISON','',f'Verdict: {verdict}',f'Pages: {len(compare)}',f'Pages with disagreement: {int((compare.different_pixels>0).sum())}',f'Total disagreement pixels: {int(compare.different_pixels.sum())}',f'Outside-M3 disagreement pixels: {int(compare.outside_M3_disagreement.sum())}',f'Policy ID: {policy_id}','','Notebook 09 remained decision authority. No threshold/model/policy mutation occurred.']
(OUT/'executive_summary.txt').write_text(chr(10).join(summary))
files=[OUT/'executive_summary.txt',OUT/'notebook55_verdict.json',A/'roster_audit.json',T/'shadow_page_comparison.csv',T/'shadow_review_queue.csv']+[p for p in V.glob('*.png')]
rows=[]
for src in files:
    require(src.exists(),f'Missing upload artefact {src}'); dst=U/src.name; shutil.copy2(src,dst); rows.append({'file':dst.name,'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(rows).to_csv(U/'upload_manifest.csv',index=False); (U/'README.txt').write_text('Notebook 55 V8.8D shadow evidence. Notebook 09 remained authoritative.'+chr(10))
zp=U/'notebook55_v8_8d_upload_bundle.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for p in U.iterdir():
        if p.is_file() and p!=zp:z.write(p,p.name)
require(zp.stat().st_size>0,'Upload ZIP failed'); print('Upload:',zp)
